In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from models.gbdt_uplift_model import TwoStageGradientBoostingUpliftClassifier
from utils.metrics import qini_score

DATASET_PATH = os.path.join(os.getcwd(), "data")

# --- 1. COPY LOGIC TÍNH TOÁN CỦA TÁC GIẢ (Từ visualization.py) ---

def get_mse_author_logic(y_true, t_true, pred_uplift):
    """
    Tính MSE theo cách của tác giả:
    So sánh dự đoán (pred_uplift) với 'effect' thật (chỉ có trong Synthetic Data).
    Công thức: (y_true - pred_uplift)^2 trung bình trên từng treatment (aka mean())
    y_true: được coi là các giá trị thực (numeric values) đại diện cho cường độ tác động
        1: chắc chắn mua
        0: không quan tâm (mua hoặc không mua bất kể có coupon hay không)
        -1: sleeping dogs: Ghét bị làm phiền (nhận coupon -> chắc chắn ko mua)
    Returns: Array of MSE cho từng treatment
    """
    # Trong Synthetic data, t_true có nhiều treatment (1, 2, 3...)
    # Hàm này tính MSE cho từng treatment
    
    n_treatments = t_true.max()
    mse_scores = []
    
    for t in range(1, n_treatments + 1):
        # Lấy các dòng thuộc treatment t
        mask = (t_true == t)
        
        if np.sum(mask) == 0:
            mse_scores.append(0.0)
            continue
            
        y_effect_true = y_true[mask] # Đây phải là cột 'effect' (True Uplift)
        
        # Nếu pred_uplift là 1 cột (binary), ta dùng trực tiếp
        # Nếu pred_uplift là nhiều cột (multi-treatment), ta lấy cột t-1
        if pred_uplift.ndim > 1 and pred_uplift.shape[1] >= t:
             p_pred = pred_uplift[mask, t-1]
        else:
             p_pred = pred_uplift[mask] # Binary case
             
        # Tính MSE
        mse_val = ((y_effect_true - p_pred) ** 2).mean()
        mse_scores.append(mse_val)
        
    return np.array(mse_scores)

# --- 2. HÀM CHẠY THỰC NGHIỆM 5 FOLDS ---

def run_5fold_experiment(dataset_alias, model_params):
    print(f"Running Experiment on {dataset_alias} (5 Folds)...")
    
    auuc_list = [] # Lưu AUUC của 5 folds
    mse_list = []  # Lưu MSE của 5 folds (nếu có)
    
    # Giả lập 5 folds bằng cách load các file dữ liệu tác giả đã tạo sẵn
    # Folder: data/synth1_0, data/synth1_1, ...
    
    for i in range(5):
        print(f"  > Processing Fold {i}...")
        
        # Load Data
        folder = os.path.join(DATASET_PATH, f'{dataset_alias}_{i}')
        if not os.path.exists(folder):
            print(f"    [!] Missing data for fold {i}. Run data_preparation.py first.")
            continue
            
        train = joblib.load(os.path.join(folder, 'train.pkl'))
        test = joblib.load(os.path.join(folder, 'test.pkl'))
        
        # Xử lý dữ liệu Synthetic (có cột 'effect' thật để tính MSE)
        has_effect = 'effect' in test
        
        # Lấy Treatment 1 làm ví dụ (Binary Case: Treat 1 vs Control)
        # Trong code tác giả, họ loop qua tất cả treatment (multi)
        # Ở đây ta demo với Treatment 1 (hillstrom: 1 men, 2 women) (synthesis: 1-6)
        target_treat_id = 1
        
        # Lọc dữ liệu Train (Control + Treat 1)
        mask_tr = np.isin(train['t'], [0, target_treat_id])
        X_tr = train['X'][mask_tr]
        y_tr = train['y'][mask_tr]
        t_tr = (train['t'][mask_tr] == target_treat_id).astype(int)
        
        # Train Model
        model = TwoStageGradientBoostingUpliftClassifier(**model_params)
        model.fit(X_tr, y_tr, t_tr)
        
        # Lọc dữ liệu Test
        mask_te = np.isin(test['t'], [0, target_treat_id])
        X_te = test['X'][mask_te]
        y_te = test['y'][mask_te]
        t_te = (test['t'][mask_te] == target_treat_id).astype(int)
        
        # Predict
        uplift_pred = model.predict(X_te)
        
        # --- TÍNH METRIC ---
        
        # 1. AUUC (Qini)
        _, score = qini_score(y_te, uplift_pred, t_te)
        auuc_list.append(score)
        
        # 2. MSE (Chỉ tính được nếu biết Uplift thật - Synthetic Data)
        if has_effect:
            # Lấy cột effect thật của test set
            effect_true = test['effect'][mask_te]
            # Tính MSE: Mean((True - Pred)^2)
            mse_val = ((effect_true - uplift_pred)**2).mean()
            mse_list.append(mse_val)
            
    return np.array(auuc_list), np.array(mse_list)

# --- 3. TẠO BẢNG KẾT QUẢ (FORMAT GIỐNG TÁC GIẢ) ---

def format_result(values):
    if len(values) == 0:
        return "N/A"
    # Format: Mean ± Std
    mean = np.mean(values)
    std = np.std(values, ddof=1) # ddof=1 cho sample std
    return f"{mean:.4f} ± {std:.4f}"

if __name__ == '__main__':
    # Cấu hình Model giống "GBDT Average" (weight=0.5)
    params = {
        'learning_rate': 0.05,
        'max_depth': 6,
        'n_estimators': 300,
        'uplift_ensemble_weight': 0.5,
        'verbose': False
    }
    
    # 1. Chạy trên Synth1 (Có MSE)
    auucs, mses = run_5fold_experiment('synth1', params)
    
    # 2. Tạo DataFrame kết quả
    results = {
        'Model': ['My GBDT Implementation'],
        'AUUC (Treat 1)': [format_result(auucs)],
        'MSE (Treat 1)': [format_result(mses)]
    }
    
    df = pd.DataFrame(results)
    
    print("\n=== REPLICATION RESULTS ===")
    print(df.to_string(index=False))

d:\projects\ml\gbdt_uplift\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running Experiment on synth1 (5 Folds)...
  > Processing Fold 0...
  > Processing Fold 1...
  > Processing Fold 2...
  > Processing Fold 3...
  > Processing Fold 4...

=== REPLICATION RESULTS ===
                 Model  AUUC (Treat 1)   MSE (Treat 1)
My GBDT Implementation 0.1010 ± 0.0043 0.0000 ± 0.0000


In [3]:
params = {
    'learning_rate': 0.05,
    'max_depth': 6,
    'n_estimators': 300,
    'uplift_ensemble_weight': 0.5,
    'verbose': False
}

# 1. Chạy trên Hillstrom 
auucs, mses = run_5fold_experiment('hillstrom', params)

# 2. Tạo DataFrame kết quả
results = {
    'Model': ['My GBDT Implementation'],
    'AUUC (Women)': [format_result(auucs)],
    'MSE (Women)': [format_result(mses)]
}

df = pd.DataFrame(results)

print("\n=== RESULTS ===")
print(df.to_string(index=False))

Running Experiment on hillstrom (5 Folds)...
  > Processing Fold 0...
  > Processing Fold 1...
  > Processing Fold 2...
  > Processing Fold 3...
  > Processing Fold 4...

=== RESULTS ===
                 Model    AUUC (Women) MSE (Women)
My GBDT Implementation 0.0525 ± 0.0030         N/A
